In [6]:
import pandas as pd
import numpy as np
from pathlib import Path
from featurewiz import featurewiz
from sklearn.model_selection import train_test_split

current_dir = Path.cwd()
project_root = current_dir.parents[2]
path_data = project_root / 'DATA_PPMI/Results/MODEL_DATA/data_BL_V08.csv'
data = pd.read_csv(path_data)
data.head()

,PATNO,EVENT_ID,ENRLLRRK2,ENRLGBA,COHORT_DEFINITION,SEX,RAWHITE,EDUCYRS,AGE_AT_VISIT,MCAALTTM,...,NP3PTRML,NP3KTRMR,NP3KTRML,NP3RTARU,NP3RTALU,NP3RTARL,NP3RTALL,NP3RTALJ,NP3RTCON,NHY
0,3003,BL,0,0,Parkinson's Disease,0.0,1.0,16.0,56.7,1.0,...,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0
1,3003,V04,0,0,Parkinson's Disease,0.0,1.0,16.0,57.7,1.0,...,0.0,2.0,0.0,2.0,0.0,0.0,0.0,0.0,1.0,2.0
2,3003,V06,0,0,Parkinson's Disease,0.0,1.0,16.0,58.8,1.0,...,1.0,2.0,1.0,2.0,0.0,0.0,0.0,0.0,1.0,2.0
3,3003,V08,0,0,Parkinson's Disease,0.0,1.0,16.0,59.7,0.0,...,0.0,2.0,1.0,2.0,0.0,0.0,0.0,0.0,1.0,2.0
4,3018,BL,0,0,Parkinson's Disease,0.0,1.0,16.0,60.5,1.0,...,0.0,0.0,0.0,2.0,0.0,1.0,0.0,0.0,1.0,2.0


In [7]:
# ============================================
# SULOV (correlation-based) + Recursive XGBoost
# End-to-end pipeline (NO leakage with CV) - ONLY SMOTE
# ============================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

from xgboost import XGBClassifier


# -----------------------------
# 1) Load data
# -----------------------------
current_dir = Path.cwd()
project_root = current_dir.parents[2]  # ajusta si tu estructura cambia

path_data = project_root / "DATA_PPMI/Results/MODEL_DATA/data_bl_v08.csv"
data = pd.read_csv(path_data)


# -----------------------------
# 2) Define target (STAGE) from NHY
# -----------------------------
def HY_classification(nhy):
    if nhy == 0:
        return 0  # Healthy
    elif nhy in [1, 2]:
        return 1  # Early Stage
    else:
        return 2  # Advanced Stage

data["STAGE"] = data["NHY"].apply(HY_classification)


# -----------------------------
# 3) Build y (target) for a single visit, indexed by PATNO
#    Here: using V08
# -----------------------------
y = (
    data.loc[data["EVENT_ID"] == "V08", ["PATNO", "STAGE"]]
        .drop_duplicates("PATNO")
        .set_index("PATNO")["STAGE"]
)


# -----------------------------
# 4) Build X as statistical features across visits per PATNO
#    (mean/min/max/var/std)
# -----------------------------
drop_cols = ["NHY", "STAGE", "COHORT_DEFINITION", "EVENT_ID"]
data2 = data.drop(columns=[c for c in drop_cols if c in data.columns])

data2_num = data2.select_dtypes(include="number")
if "PATNO" not in data2_num.columns:
    raise ValueError("PATNO no está en data2_num. Revisa columnas numéricas / drop_cols.")

df_grouped = data2_num.groupby("PATNO")

df_mean = df_grouped.mean().add_suffix("_mean")
df_min  = df_grouped.min().add_suffix("_min")
df_max  = df_grouped.max().add_suffix("_max")
df_var  = df_grouped.var().add_suffix("_var")
df_std  = df_grouped.std().add_suffix("_std")

X = pd.concat([df_mean, df_min, df_max, df_var, df_std], axis=1)

# Align X and y by PATNO intersection
common = X.index.intersection(y.index)
X = X.loc[common].copy()
y = y.loc[common].copy()

if X.shape[0] != y.shape[0]:
    raise ValueError(f"X rows ({X.shape[0]}) != y rows ({y.shape[0]}). Revisa alineación por PATNO.")
if y.isna().any():
    raise ValueError("Hay NaNs en y (STAGE). Revisa tu construcción de target.")
if X.isna().any().any():
    X = X.fillna(X.median(numeric_only=True))


# -----------------------------
# 5) Train/test split (NO leakage)
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# -----------------------------
# 6) SULOV "real" (correlation filter) - do on TRAIN only
# -----------------------------
def sulov_corr_filter(X: pd.DataFrame, corr_limit: float = 0.90):
    """
    Correlation filter:
    - Compute absolute correlation among features
    - If corr > corr_limit, drop one feature (keep higher variance)
    """
    Xn = X.copy()
    corr = Xn.corr(numeric_only=True).abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

    variances = Xn.var(numeric_only=True)
    to_drop = set()

    for col in upper.columns:
        rows = upper.index[upper[col] > corr_limit].tolist()
        for row in rows:
            if row in to_drop or col in to_drop:
                continue
            drop = row if variances[row] < variances[col] else col
            to_drop.add(drop)

    kept = [c for c in Xn.columns if c not in to_drop]
    return Xn[kept], sorted(list(to_drop)), kept


corr_limit = 0.90
X_train_sulov, dropped_corr, kept_corr = sulov_corr_filter(X_train, corr_limit=corr_limit)
X_test_sulov = X_test[kept_corr]

print("\n=== SULOV (Correlation Filter) ===")
print(f"Corr limit: {corr_limit}")
print(f"Features before: {X_train.shape[1]}")
print(f"Features after : {X_train_sulov.shape[1]}")
print(f"Dropped        : {len(dropped_corr)}")


# -----------------------------
# 7) Recursive XGBoost selection (NO leakage in CV)
#    CV uses Pipeline(Scaler + SMOTE + XGB)
# -----------------------------
def recursive_xgb_select(
    X: pd.DataFrame,
    y: pd.Series,
    min_features: int = 50,
    step_fraction: float = 0.10,
    random_state: int = 42,
    n_splits: int = 5,
    smote_k_neighbors: int = 3
):
    """
    Iteratively:
      - Evaluate CV accuracy using Pipeline(Scaler + SMOTE + XGB) to avoid leakage
      - Fit helper XGB on scaled data to get feature importances (practical approximation)
      - Drop worst step_fraction features
    Keeps best feature set by CV accuracy.
    """
    X_curr = X.copy()
    best_score = -np.inf
    best_cols = X_curr.columns.tolist()

    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    while X_curr.shape[1] > min_features:
        xgb_cv = XGBClassifier(
            n_estimators=400,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            objective="multi:softprob",
            num_class=len(np.unique(y)),
            eval_metric="mlogloss",
            random_state=random_state,
            n_jobs=-1
        )

        smote = SMOTE(k_neighbors=smote_k_neighbors, random_state=random_state)

        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("smote", smote),
            ("model", xgb_cv),
        ])

        scores = cross_val_score(pipe, X_curr, y, cv=cv, scoring="accuracy")
        score = scores.mean()

        if score > best_score:
            best_score = score
            best_cols = X_curr.columns.tolist()

        # importances helper (rápido y estable)
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_curr)

        xgb_imp = XGBClassifier(
            n_estimators=400,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            objective="multi:softprob",
            num_class=len(np.unique(y)),
            eval_metric="mlogloss",
            random_state=random_state,
            n_jobs=-1
        )
        xgb_imp.fit(X_scaled, y)
        importances = pd.Series(xgb_imp.feature_importances_, index=X_curr.columns)

        k = max(1, int(len(importances) * step_fraction))
        worst = importances.nsmallest(k).index
        X_curr = X_curr.drop(columns=worst)

        if X_curr.shape[1] <= min_features:
            break

    return best_cols, best_score


min_features = 50
step_fraction = 0.10

best_cols, best_cv = recursive_xgb_select(
    X_train_sulov, y_train,
    min_features=min_features,
    step_fraction=step_fraction,
    random_state=42,
    n_splits=5,
    smote_k_neighbors=3
)

print("\n=== Recursive XGBoost Selection ===")
print(f"Min features: {min_features}")
print(f"Step fraction: {step_fraction}")
print(f"Best CV accuracy: {best_cv:.4f}")
print(f"Final feature count: {len(best_cols)}")


# -----------------------------
# 8) Train final model and evaluate on test (NO leakage)
#    Fit pipeline on TRAIN, test on TEST
# -----------------------------
final_xgb = XGBClassifier(
    n_estimators=600,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    objective="multi:softprob",
    num_class=len(np.unique(y_train)),
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

final_smote = SMOTE(k_neighbors=3, random_state=42)

final_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("smote", final_smote),
    ("model", final_xgb),
])

X_train_final = X_train_sulov[best_cols]
X_test_final  = X_test_sulov[best_cols]

final_pipe.fit(X_train_final, y_train)
y_pred = final_pipe.predict(X_test_final)

print("\n=== Test Evaluation ===")
print("Test accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred))

print("\n=== Selected Features (final) ===")
print(best_cols)


=== SULOV (Correlation Filter) ===
Corr limit: 0.9
Features before: 955
Features after : 674
Dropped        : 281

=== Recursive XGBoost Selection ===
Min features: 50
Step fraction: 0.1
Best CV accuracy: 0.9340
Final feature count: 116

=== Test Evaluation ===
Test accuracy: 0.9230769230769231

Confusion matrix:
 [[80  2  0]
 [ 8 87  0]
 [ 0  4  1]]

Classification report:
               precision    recall  f1-score   support

           0       0.91      0.98      0.94        82
           1       0.94      0.92      0.93        95
           2       1.00      0.20      0.33         5

    accuracy                           0.92       182
   macro avg       0.95      0.70      0.73       182
weighted avg       0.93      0.92      0.92       182


=== Selected Features (final) ===
['ENRLGBA_mean', 'RAWHITE_mean', 'AGE_AT_VISIT_mean', 'MCARHINO_mean', 'MCAFDS_mean', 'MCABDS_mean', 'MCASER7_mean', 'MCAREC1_mean', 'MCAREC2_mean', 'SCAU15_mean', 'SCAU18_mean', 'GDSHLPLS_mean', 'GDSWRTLS